In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
make_figures_ch4.py

Generate main figures for Chapter 4 (MDS & clustering) as PNG and PDF
for insertion into Word.

- Fig 4.2.2.1 : Eigenvalues (OH vs FP)
- Fig 4.2.3.1 : MDS scatter (OH, linear_top3)
- Fig 4.2.3.2 : MDS scatter (FP, linear_top3)
- Fig 4.2.4.3 : MDS + clusters (FP, linear_cumeig)
- Fig 4.2.4.4 : MDS + clusters (OH, linear_cumeig)
- Fig 4.2.5.1 : ARI / NMI / AMI bar plot (sign vs signless)

NOTE:
- All labels, axis titles and legends are in English (for thesis).
- Colors/styles are simple and Word-friendly.
"""

import os
import glob
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 0. User settings
# ============================================================

# --- root directory for this analysis (change if needed) ---
ROOT = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# --- STEP3.2 run ID (folder name after 'run_') -------------
RUN_ID_STEP32 = "20251130_153500"  # <- 例。実際の run_XXXX を指定

# --- STEP3.3 / summary for ARI/NMI/AMI (if you have it) ----
# 例: sub/STEP3.3_U25_20251027_correct/.../OHFP_contrast_summary.csv
ARI_SUMMARY_CSV = None  # パスが決まったら文字列で指定

# --- output directory for figures --------------------------
OUT_DIR = os.path.join(ROOT, "figures_tables", "STEP3.2_Chapter4_main")
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 1. Utility functions
# ============================================================

def find_one_csv(pattern: str) -> str:
    """Return a single csv path matching the glob pattern.
    Raise error if not exactly one file is found.
    """
    paths = glob.glob(pattern)
    if len(paths) == 0:
        raise FileNotFoundError(f"[ERROR] No file found for pattern: {pattern}")
    if len(paths) > 1:
        print(f"[WARN] Multiple files found for pattern: {pattern}")
        print("       Using the first one:")
        for p in paths:
            print("        -", p)
    return sorted(paths)[0]


def save_figure(fig: plt.Figure, basename: str, dpi: int = 300):
    """Save matplotlib figure as PNG and PDF."""
    png_path = os.path.join(OUT_DIR, f"{basename}.png")
    pdf_path = os.path.join(OUT_DIR, f"{basename}.pdf")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=dpi, bbox_inches="tight")
    print(f"[SAVED] {png_path}")
    print(f"[SAVED] {pdf_path}")


# ============================================================
# 2. Fig 4.2.2.1 : Eigenvalues (OH vs FP)
# ============================================================

def plot_eigen_oh_fp():
    """
    Fig 4.2.2.1
    Use eigen CSVs for OH and FP (linear MDS).
    Expect columns like: 'axis', 'eigen', 'ratio', 'cumulative_ratio'
    """
    base_eigen_dir = os.path.join(ROOT, "sub", "02_mds_STEP3.2_signlessCorr",
                                  f"run_{RUN_ID_STEP32}", "eigen")

    # ---- OH ----
    # 例: MDS_eigen_linear_variables_OH_20251130_153500.csv
    pattern_oh = os.path.join(base_eigen_dir, "OH", "MDS_eigen_linear_*OH*.csv")
    path_oh = find_one_csv(pattern_oh)
    eig_oh = pd.read_csv(path_oh)

    # ---- FP ----
    pattern_fp = os.path.join(base_eigen_dir, "FP", "MDS_eigen_linear_*FP*.csv")
    path_fp = find_one_csv(pattern_fp)
    eig_fp = pd.read_csv(path_fp)

    # シンプルに「相対寄与率」と「累積寄与率」を折れ線で
    fig, ax = plt.subplots(figsize=(6, 4))

    x_oh = eig_oh["axis"]
    ax.plot(x_oh, eig_oh["ratio"], marker="o", linestyle="-", label="OH (ratio)")
    ax.plot(x_oh, eig_oh["cumulative_ratio"], marker="o", linestyle="--",
            label="OH (cumulative)")

    x_fp = eig_fp["axis"]
    ax.plot(x_fp, eig_fp["ratio"], marker="s", linestyle="-", label="FP (ratio)")
    ax.plot(x_fp, eig_fp["cumulative_ratio"], marker="s", linestyle="--",
            label="FP (cumulative)")

    ax.set_xlabel("MDS axis")
    ax.set_ylabel("Relative / cumulative eigenvalue")
    ax.set_title("Eigenvalue ratios of MDS (OH / FP, linear)")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="best", frameon=False)

    save_figure(fig, "Fig_4.2.2.1_MDS_eigen_OH_FP")
    plt.close(fig)


# ============================================================
# 3. Fig 4.2.3.1 & 4.2.3.2 : MDS scatter (OH / FP, top3)
# ============================================================

def load_mds_coords(dataset: str, mode: str = "linear_top3") -> pd.DataFrame:
    """Load MDS coordinates for variables.

    dataset: 'OH' or 'FP'
    mode   : 'linear_top3' / 'linear_cumeig' etc.
    """
    base_dir = os.path.join(ROOT, "sub", "02_mds_STEP3.2_signlessCorr",
                            f"run_{RUN_ID_STEP32}", "variables", dataset)
    pattern = os.path.join(base_dir, f"MDScoords_{mode}_variables_{dataset}_*.csv")
    path = find_one_csv(pattern)
    df = pd.read_csv(path)
    return df


def plot_mds_scatter(df: pd.DataFrame, dataset: str, basename: str,
                     x_col: str = "Dim1", y_col: str = "Dim2"):
    """Generic 2D scatter: variables in MDS space."""
    fig, ax = plt.subplots(figsize=(5, 4))

    ax.scatter(df[x_col], df[y_col], s=10, alpha=0.7)

    ax.set_xlabel("MDS axis 1")
    ax.set_ylabel("MDS axis 2")
    ax.set_title(f"MDS scatter ({dataset}, {x_col} vs {y_col})")

    save_figure(fig, basename)
    plt.close(fig)


def make_fig_4_2_3_oh_fp():
    """
    Fig 4.2.3.1 : OH (linear_top3, Dim1×Dim2)
    Fig 4.2.3.2 : FP (linear_top3, Dim1×Dim2)
    """
    # OH
    df_oh = load_mds_coords("OH", mode="linear_top3")
    if "Dim1" not in df_oh.columns or "Dim2" not in df_oh.columns:
        # RDKit/MDSの実装によっては 'Axis1','Axis2' などの可能性もある
        print("[WARN] Column names not found (Dim1/Dim2). "
              "Please adjust x_col/y_col manually.")
    plot_mds_scatter(df_oh, "OH", "Fig_4.2.3.1_MDS_scatter_OH",
                     x_col="Dim1", y_col="Dim2")

    # FP
    df_fp = load_mds_coords("FP", mode="linear_top3")
    plot_mds_scatter(df_fp, "FP", "Fig_4.2.3.2_MDS_scatter_FP",
                     x_col="Dim1", y_col="Dim2")


# ============================================================
# 4. Fig 4.2.4.3 & 4.2.4.4 : MDS + cluster color (cumeig)
# ============================================================

def load_cluster_labels(dataset: str,
                        mode: str = "linear_cumeig",
                        criterion: str = "silhouette") -> pd.DataFrame:
    """Load cluster labels for variables.

    Expected columns: the first column = variable ID (same as in MDS coords),
    and one column 'cluster' or similar.
    """
    base_dir = os.path.join(ROOT, "sub", "03_clustering_STEP3.2_signlessCorr",
                            f"run_{RUN_ID_STEP32}", "variables", dataset, "labels")

    pattern = os.path.join(base_dir,
                           f"ClusterAssign_{mode}_{criterion}_variables_{dataset}_*.csv")
    path = find_one_csv(pattern)
    df = pd.read_csv(path)
    return df


def merge_coords_labels(df_coords: pd.DataFrame,
                        df_labels: pd.DataFrame) -> pd.DataFrame:
    """
    Merge MDS coords and cluster labels.
    We try to guess the name of the ID column:
    - 'var_id' or 'variable' or first column.
    """
    # guess ID column
    # (safe: use first column of each as ID)
    coords_id_col = df_coords.columns[0]
    labels_id_col = df_labels.columns[0]

    merged = pd.merge(df_coords, df_labels,
                      left_on=coords_id_col, right_on=labels_id_col,
                      how="inner", suffixes=("_coords", "_cl"))

    # guess cluster column
    cluster_cols = [c for c in merged.columns if "cluster" in c.lower()]
    if len(cluster_cols) == 0:
        raise ValueError("No cluster column found in merged dataframe.")
    merged["cluster"] = merged[cluster_cols[0]]
    return merged


def plot_mds_clusters(merged: pd.DataFrame, dataset: str, basename: str,
                      x_col: str = "Dim1", y_col: str = "Dim2"):
    """Scatter with color by cluster."""
    fig, ax = plt.subplots(figsize=(5, 4))

    clusters = np.unique(merged["cluster"])
    for cl in clusters:
        sub = merged[merged["cluster"] == cl]
        ax.scatter(sub[x_col], sub[y_col], s=10, alpha=0.8, label=f"Cluster {cl}")

    ax.set_xlabel("MDS axis 1")
    ax.set_ylabel("MDS axis 2")
    ax.set_title(f"Variable clusters in MDS space ({dataset}, cumeig)")
    ax.legend(loc="best", frameon=False, fontsize=8)

    save_figure(fig, basename)
    plt.close(fig)


def make_fig_4_2_4_clusters():
    """
    Fig 4.2.4.3 : FP clusters (linear_cumeig)
    Fig 4.2.4.4 : OH clusters (linear_cumeig)
    """
    # FP
    df_fp = load_mds_coords("FP", mode="linear_cumeig")
    df_fp_lab = load_cluster_labels("FP", mode="linear_cumeig",
                                    criterion="silhouette")
    merged_fp = merge_coords_labels(df_fp, df_fp_lab)
    plot_mds_clusters(merged_fp, "FP", "Fig_4.2.4.3_MDS_clusters_FP",
                      x_col="Dim1", y_col="Dim2")

    # OH
    df_oh = load_mds_coords("OH", mode="linear_cumeig")
    df_oh_lab = load_cluster_labels("OH", mode="linear_cumeig",
                                    criterion="silhouette")
    merged_oh = merge_coords_labels(df_oh, df_oh_lab)
    plot_mds_clusters(merged_oh, "OH", "Fig_4.2.4.4_MDS_clusters_OH",
                      x_col="Dim1", y_col="Dim2")


# ============================================================
# 5. Fig 4.2.5.1 : ARI / NMI / AMI (sign vs signless)
# ============================================================

def make_fig_4_2_5_ari():
    """
    Fig 4.2.5.1
    ARI / NMI / AMI between
      - sign-correlation-based clustering
      - signlessCorrelation-based clustering

    Expect summary CSV like:

        metric, dataset, value
        ARI, OH, 0.93
        NMI, OH, 0.90
        AMI, OH, 0.89
        ARI, FP, 0.80
        ...

    If your actual format is different, please adjust.
    """
    if ARI_SUMMARY_CSV is None:
        print("[INFO] ARI_SUMMARY_CSV is not specified. "
              "Skip Fig 4.2.5.1.")
        return

    df = pd.read_csv(ARI_SUMMARY_CSV)

    # basic reshape: metric x dataset
    # assume columns: metric, dataset, value
    pivot = df.pivot(index="metric", columns="dataset", values="value")

    metrics = list(pivot.index)
    datasets = list(pivot.columns)

    x = np.arange(len(metrics))
    width = 0.35 if len(datasets) == 2 else 0.8 / len(datasets)

    fig, ax = plt.subplots(figsize=(5, 4))

    for i, ds in enumerate(datasets):
        vals = pivot[ds].values
        ax.bar(x + i * width, vals, width, label=ds)

    ax.set_xticks(x + width * (len(datasets) - 1) / 2)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Agreement score")
    ax.set_title("Agreement between sign and signlessCorr clustering")
    ax.legend(loc="best", frameon=False)

    save_figure(fig, "Fig_4.2.5.1_cluster_agreement")
    plt.close(fig)


# ============================================================
# 6. Main
# ============================================================

def main():
    print("=== Generate Chapter 4 figures (STEP3.2) ===")
    print(f"ROOT        : {ROOT}")
    print(f"RUN_ID      : {RUN_ID_STEP32}")
    print(f"OUTPUT DIR  : {OUT_DIR}")
    print("-----------------------------------------------")

    # Fig 4.2.2.1
    print("\n[1] Fig 4.2.2.1 : Eigenvalues (OH / FP)")
    plot_eigen_oh_fp()

    # Fig 4.2.3.1 & 4.2.3.2
    print("\n[2] Fig 4.2.3.1 & 4.2.3.2 : MDS scatter (OH / FP)")
    make_fig_4_2_3_oh_fp()

    # Fig 4.2.4.3 & 4.2.4.4
    print("\n[3] Fig 4.2.4.3 & 4.2.4.4 : MDS clusters (OH / FP, cumeig)")
    make_fig_4_2_4_clusters()

    # Fig 4.2.5.1
    print("\n[4] Fig 4.2.5.1 : ARI / NMI / AMI (if summary is provided)")
    make_fig_4_2_5_ari()

    print("\nDone.")


if __name__ == "__main__":
    main()


=== Generate Chapter 4 figures (STEP3.2) ===
ROOT        : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127
RUN_ID      : 20251130_153500
OUTPUT DIR  : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/figures_tables/STEP3.2_Chapter4_main
-----------------------------------------------

[1] Fig 4.2.2.1 : Eigenvalues (OH / FP)


FileNotFoundError: [ERROR] No file found for pattern: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/eigen/OH/MDS_eigen_linear_*OH*.csv